# Planting-window pipeline — run in Google Colab

Crop-specific dekadal planting-window estimation (GHA / ICPAC) on Google Earth Engine.

**Before you start:** your Google account must be registered for Earth Engine and you need a
GEE **cloud project** (here `ee-manzikye`). Outputs export to **Google Drive → `planting_outputs/`**.

> ⚠️ Colab does **not** raise your compute quota — that is tied to the project, not the runtime.
> If the project is in noncommercial *restricted mode*, heavy jobs (WRSI) will still be throttled here.

## 1. Install dependencies
(Colab already has `earthengine-api`, but this pins current versions.)

### Stage 0 · What this notebook is for

**Use this one to launch batch exports, not to explore.** It uploads the pipeline as a zip, calls
`run.py`, and then watches the Earth Engine task queue. Nothing is computed in the notebook itself:
`run.py` submits **batch exports** that run on Google's servers and land in Drive, so you can close the
browser once they are submitted.

For interactive work with maps, use `01_planting_window` through `04_flooding_waterlogging`, which
import the pipeline from Drive instead of uploading a zip.

**Expected output.** Pip installs quietly. Warnings about resolver conflicts are normal.

**A quota point that catches people out.** Colab does not raise your Earth Engine compute quota. Quota
belongs to the **cloud project**, not to the runtime. A heavy WRSI run will be throttled here exactly
as it would anywhere else, and paying for more Colab compute changes nothing.

In [1]:
!pip -q install "earthengine-api>=1.4.0" pyyaml pandas

## 2. Authenticate + initialise Earth Engine
Run the cell, click the link, paste the token. Set `PROJECT` to your GEE cloud project.

### Stage 1 · Earth Engine sign-in

**Expected output.** `EE ready: ok`.

**Choose the project deliberately.** The **export queue is per project**. `ee-manzikye` has left
batches sitting in READY for hours, including a Somalia WRSI export that had to be cancelled. For any
run that matters, set `PROJECT = "indigo-proxy-484220-q8"`, whose queue has been reliable. Compute is
identical; only the queue differs.

In [3]:
import ee

PROJECT = "ee-manzikye"   # <-- your GEE cloud project id

ee.Authenticate()          # opens an auth link the first time
ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

EE ready: ok


## 3. Get the pipeline code into Colab
Upload `planting_pipeline.zip` (the file I built for you) using the cell below, then it unzips
and `cd`s into it.

### Stage 2 · Get the code into the runtime

**What runs.** Uploads `planting_pipeline.zip`, unpacks it and changes into it. The upload widget is
a browser control, so this cell will sit and wait until you choose a file.

**Expected output.** `cwd: /content/planting_pipeline` and a file listing that includes `run.py`,
`src`, `config` and the validation CSVs. If `config` is missing, the zip was made from inside the wrong
folder and `run.py` will fail on the crop calendar.

**The alternative, which is usually better.** Keep the folder on Drive and mount it, as the module
notebooks do. A zip is a snapshot: any fix you make later has to be re-zipped and re-uploaded.

In [4]:
import zipfile, os
from google.colab import files

up = files.upload()                 # choose planting_pipeline.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall(".")
os.chdir("planting_pipeline")
print("cwd:", os.getcwd())
print("files:", sorted(os.listdir()))

ModuleNotFoundError: No module named 'google.colab'

## 4. Run one product
`run.py` starts Earth Engine batch exports (they run on Google's servers, land in Drive).
Change `--country` / `--crop` / `--year`, or drop `--country`/`--crop` to run everything viable for the year.
Add `--mask-asset users/you/your_mask` for a crop-specific mask (otherwise WorldCereal is used).

### Stage 3 · Submit the run

**What this stage does.** `run.py` builds the graph for one product and submits Earth Engine batch
exports. Arguments:

| Argument | Effect |
|---|---|
| `--year` | the planting year |
| `--country`, `--crop` | one product. Drop both to submit everything viable for that year |
| `--mask-asset` | a crop-specific mask image. Without it the run falls back to WorldCereal |

**Use `--mask-asset`.** WorldCereal is a single 2021 season and does not separate the crops. The
crop-type masks built by `crop_type_mask_colab.ipynb` are the intended input, for example
`projects/<project>/assets/crop_type_mask/croptype_KE_100m` selecting `mask_maize`. The difference is
not cosmetic: the mask decides which pixels every statistic is computed over.

**Expected output.** One line per submitted task. Submission takes seconds; the exports themselves
take from minutes to hours.

**Three exports per product**, described in stage 5.

In [ ]:
os.environ["EE_PROJECT"] = PROJECT
!EE_PROJECT=$PROJECT python run.py --year 2024 --country Kenya --crop maize

## 5. Monitor the export tasks
Re-run this cell to refresh. States: PENDING → RUNNING → SUCCEEDED / FAILED / CANCELLED.

### Stage 4 · Watch the queue

**What this stage does.** Lists the tasks whose description matches the filter, keeping only the latest
attempt of each name. Re-run the cell to refresh; it does not poll on its own.

**States.** `PENDING → RUNNING → SUCCEEDED`, or `FAILED` / `CANCELLED`. A failure prints its message.

**Change the filter.** It is hard-coded to `"Kenya_maize"`. If you ran another country, nothing will
print and the run will look as though it never started.

**How long is too long.** Minutes in PENDING is normal. **More than about 20 minutes with nothing
moving to RUNNING is a stalled queue, not a slow one.** That has happened repeatedly in `ee-manzikye`.
Cancel the tasks, switch the project, and resubmit; waiting does not clear it.

**Common failures.**

* *Too many pixels* — the AOI or the scale is too large. Export at 250 m, or tile the AOI.
* *Computation timed out* — the graph is too deep for one task, usually from running the whole
  country with the fused green-up. Split by season or by region.
* *Asset not found* — a `--mask-asset` path that does not exist, or one owned by a different project.

In [ ]:
seen = {}
for op in ee.data.listOperations():
    md = op.get("metadata", {})
    d = md.get("description", "")
    if "Kenya_maize" in d:                    # adjust filter to your run
        ct = md.get("createTime", "")
        if d not in seen or ct > seen[d][0]:   # keep the latest task per name
            seen[d] = (ct, md.get("state", "?"), op.get("error", {}).get("message", ""))
for d in sorted(seen):
    ct, st, err = seen[d]
    print(f"{st:10} {d}" + (f"  -> {err}" if err else ""))

## 6. Where the outputs are
When tasks show **SUCCEEDED**, files are in your **Google Drive → `planting_outputs/`**:
- `planting_<country>_<crop>_<season>_<year>` — GeoTIFF, per-pixel planting dekad
- `wrsi_<...>` — GeoTIFF (WRSI + deficit mm + crop-performance class)
- `<...>_zonal` — CSV, admin-1 modal / P10 / P50 / P90 planting dekad

You can also watch tasks at https://code.earthengine.google.com/tasks